In [7]:
import numpy as np
import pandas as pd
from sklearn.neighbors import BallTree

In [8]:
hdb = pd.read_csv("../data/cleaned/hdb_cleaned.csv")
hawker_centres = pd.read_csv("../data/cleaned/hawker_centres_cleaned.csv")
bus_stops = pd.read_csv("../data/cleaned/bus_stops_cleaned.csv")
mall = pd.read_csv("../data/cleaned/mall_cleaned.csv")
mrt = pd.read_csv("../data/cleaned/mrt_cleaned.csv")
sch = pd.read_csv("../data/cleaned/sch_cleaned.csv")
rpi = pd.read_csv("../data/cleaned/rpi_cleaned.csv")

In [9]:
# left join rpi to hdb to get real prices on quarter and year
hdb = hdb.merge(rpi, on=["quarter", "year"], how="left")
hdb.head()

,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,remaining_lease,resale_price,year,quarter,remaining_lease_years,mid_storey,full_address,log_resale_price,lat,lon,Index
0,2017-01-01,ANG MO KIO,2 ROOM,406,ANG MO KIO AVE 10,10 TO 12,44.0,Improved,1979,61 years 04 months,232000.0,2017,1,61.333333,11.0,406 ANG MO KIO AVE 10,12.354493,1.362005,103.853880,133.9
1,2017-01-01,ANG MO KIO,3 ROOM,108,ANG MO KIO AVE 4,01 TO 03,67.0,New Generation,1978,60 years 07 months,250000.0,2017,1,60.583333,2.0,108 ANG MO KIO AVE 4,12.429216,1.370966,103.838202,133.9
2,2017-01-01,ANG MO KIO,3 ROOM,602,ANG MO KIO AVE 5,01 TO 03,67.0,New Generation,1980,62 years 05 months,262000.0,2017,1,62.416667,2.0,602 ANG MO KIO AVE 5,12.476100,1.380709,103.835368,133.9
3,2017-01-01,ANG MO KIO,3 ROOM,465,ANG MO KIO AVE 10,04 TO 06,68.0,New Generation,1980,62 years 01 month,265000.0,2017,1,62.083333,5.0,465 ANG MO KIO AVE 10,12.487485,1.366201,103.857201,133.9
4,2017-01-01,ANG MO KIO,3 ROOM,601,ANG MO KIO AVE 5,01 TO 03,67.0,New Generation,1980,62 years 05 months,265000.0,2017,1,62.416667,2.0,601 ANG MO KIO AVE 5,12.487485,1.381041,103.835132,133.9


In [10]:
# calculate real prices
SQM_TO_SQFT = 10.7639
hdb["floor_area_sqft"] = hdb["floor_area_sqm"] * SQM_TO_SQFT

hdb["real_price"] = hdb["resale_price"] / (hdb["Index"] / 100.0)
hdb["resale_ppsf"] = hdb["resale_price"] / hdb["floor_area_sqft"]
hdb["real_ppsf"] = hdb["real_price"] / hdb["floor_area_sqft"]

# log prices
hdb["log_real_price"] = np.log(hdb["real_price"])
hdb["log_resale_ppsf"] = np.log(hdb["resale_ppsf"])
hdb["log_real_ppsf"] = np.log(hdb["real_ppsf"])
hdb.head()

,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,remaining_lease,...,lat,lon,Index,floor_area_sqft,real_price,resale_ppsf,real_ppsf,log_real_price,log_resale_ppsf,log_real_ppsf
0,2017-01-01,ANG MO KIO,2 ROOM,406,ANG MO KIO AVE 10,10 TO 12,44.0,Improved,1979,61 years 04 months,...,1.362005,103.853880,133.9,473.6116,173263.629574,489.852867,365.834852,12.062570,6.194105,5.902182
1,2017-01-01,ANG MO KIO,3 ROOM,108,ANG MO KIO AVE 4,01 TO 03,67.0,New Generation,1978,60 years 07 months,...,1.370966,103.838202,133.9,721.1813,186706.497386,346.653470,258.889821,12.137293,5.848326,5.556403
2,2017-01-01,ANG MO KIO,3 ROOM,602,ANG MO KIO AVE 5,01 TO 03,67.0,New Generation,1980,62 years 05 months,...,1.380709,103.835368,133.9,721.1813,195668.409261,363.292836,271.316532,12.184177,5.895209,5.603286
3,2017-01-01,ANG MO KIO,3 ROOM,465,ANG MO KIO AVE 10,04 TO 06,68.0,New Generation,1980,62 years 01 month,...,1.366201,103.857201,133.9,731.9452,197908.887229,362.048962,270.387574,12.195562,5.891779,5.599856
4,2017-01-01,ANG MO KIO,3 ROOM,601,ANG MO KIO AVE 5,01 TO 03,67.0,New Generation,1980,62 years 05 months,...,1.381041,103.835132,133.9,721.1813,197908.887229,367.452678,274.423210,12.195562,5.906595,5.614671


In [11]:
# rpi dont have 2026 data, remove 2026 data
hdb = hdb[hdb["year"] < 2026]

In [12]:
EARTH_RADIUS_KM = 6371.0

def add_nearest_distance_km(
    left_df, left_lat, left_lon,
    right_df, right_lat, right_lon,
    out_col
):
    # coordinates in radians: [lat, lon]
    left_coords = np.deg2rad(left_df[[left_lat, left_lon]].to_numpy())
    right_coords = np.deg2rad(right_df[[right_lat, right_lon]].to_numpy())

    tree = BallTree(right_coords, metric="haversine")
    dist_rad, _ = tree.query(left_coords, k=1)
    left_df[out_col] = dist_rad[:, 0] * EARTH_RADIUS_KM
    return left_df


In [13]:
df = hdb.copy()

df = add_nearest_distance_km(df, "lat", "lon", hawker_centres, "lat", "lon", "dist_to_nearest_hawker_km")
df = add_nearest_distance_km(df, "lat", "lon", bus_stops,      "lat", "lon", "dist_to_nearest_busstop_km")
df = add_nearest_distance_km(df, "lat", "lon", mall,           "lat", "lon", "dist_to_nearest_mall_km")
df = add_nearest_distance_km(df, "lat", "lon", sch,            "lat", "lon", "dist_to_nearest_school_km")
df = add_nearest_distance_km(df, "lat", "lon", mrt,            "lat", "lon", "dist_to_nearest_mrt_km")
df.head()

,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,remaining_lease,...,resale_ppsf,real_ppsf,log_real_price,log_resale_ppsf,log_real_ppsf,dist_to_nearest_hawker_km,dist_to_nearest_busstop_km,dist_to_nearest_mall_km,dist_to_nearest_school_km,dist_to_nearest_mrt_km
0,2017-01-01,ANG MO KIO,2 ROOM,406,ANG MO KIO AVE 10,10 TO 12,44.0,Improved,1979,61 years 04 months,...,489.852867,365.834852,12.062570,6.194105,5.902182,0.172419,0.091924,1.031166,0.514121,1.003996
1,2017-01-01,ANG MO KIO,3 ROOM,108,ANG MO KIO AVE 4,01 TO 03,67.0,New Generation,1978,60 years 07 months,...,346.653470,258.889821,12.137293,5.848326,5.556403,0.410546,0.166312,0.869079,0.242224,0.189875
2,2017-01-01,ANG MO KIO,3 ROOM,602,ANG MO KIO AVE 5,01 TO 03,67.0,New Generation,1980,62 years 05 months,...,363.292836,271.316532,12.184177,5.895209,5.603286,0.585521,0.129345,1.529046,0.777626,0.535117
3,2017-01-01,ANG MO KIO,3 ROOM,465,ANG MO KIO AVE 10,04 TO 06,68.0,New Generation,1980,62 years 01 month,...,362.048962,270.387574,12.195562,5.891779,5.599856,0.245970,0.069453,0.891974,0.697427,0.945527
4,2017-01-01,ANG MO KIO,3 ROOM,601,ANG MO KIO AVE 5,01 TO 03,67.0,New Generation,1980,62 years 05 months,...,367.452678,274.423210,12.195562,5.906595,5.614671,0.611017,0.149898,1.572922,0.782771,0.501151


In [14]:
# check for na in all col
print(df.isna().sum())


month                         0
town                          0
flat_type                     0
block                         0
street_name                   0
storey_range                  0
floor_area_sqm                0
flat_model                    0
lease_commence_date           0
remaining_lease               0
resale_price                  0
year                          0
quarter                       0
remaining_lease_years         0
mid_storey                    0
full_address                  0
log_resale_price              0
lat                           0
lon                           0
Index                         0
floor_area_sqft               0
real_price                    0
resale_ppsf                   0
real_ppsf                     0
log_real_price                0
log_resale_ppsf               0
log_real_ppsf                 0
dist_to_nearest_hawker_km     0
dist_to_nearest_busstop_km    0
dist_to_nearest_mall_km       0
dist_to_nearest_school_km     0
dist_to_

In [15]:
# housing data
merged_df = df.copy()

# school data must have: school_name, lat, lon, is_good_sch
good_sch = sch[sch["is_good_sch"] == 1].copy()

# drop missing coordinates
merged_df = merged_df.dropna(subset=["lat", "lon"]).copy()
good_sch = good_sch.dropna(subset=["lat", "lon"]).copy()

# convert to radians for haversine distance
house_coords = np.radians(merged_df[["lat", "lon"]].to_numpy())
school_coords = np.radians(good_sch[["lat", "lon"]].to_numpy())

# build nearest-neighbour tree
tree = BallTree(school_coords, metric="haversine")

# nearest good school for each house
dist, ind = tree.query(house_coords, k=1)

# convert earth-radius distance to km
earth_radius_km = 6371
merged_df["dist_to_nearest_good_school_km"] = dist[:, 0] * earth_radius_km
merged_df["nearest_good_school"] = good_sch.iloc[ind[:, 0]]["search_school"].values

# create flags
merged_df["within_1km_good_sch"] = (
    merged_df["dist_to_nearest_good_school_km"] <= 1
).astype(int)

merged_df["within_1_to_2km_good_sch"] = (
    (merged_df["dist_to_nearest_good_school_km"] > 1) &
    (merged_df["dist_to_nearest_good_school_km"] <= 2)
).astype(int)

merged_df.head()

,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,remaining_lease,...,log_real_ppsf,dist_to_nearest_hawker_km,dist_to_nearest_busstop_km,dist_to_nearest_mall_km,dist_to_nearest_school_km,dist_to_nearest_mrt_km,dist_to_nearest_good_school_km,nearest_good_school,within_1km_good_sch,within_1_to_2km_good_sch
0,2017-01-01,ANG MO KIO,2 ROOM,406,ANG MO KIO AVE 10,10 TO 12,44.0,Improved,1979,61 years 04 months,...,5.902182,0.172419,0.091924,1.031166,0.514121,1.003996,1.368382,Catholic High,0,1
1,2017-01-01,ANG MO KIO,3 ROOM,108,ANG MO KIO AVE 4,01 TO 03,67.0,New Generation,1978,60 years 07 months,...,5.556403,0.410546,0.166312,0.869079,0.242224,0.189875,0.520195,CHIJ St. Nicholas Girls',1,0
2,2017-01-01,ANG MO KIO,3 ROOM,602,ANG MO KIO AVE 5,01 TO 03,67.0,New Generation,1980,62 years 05 months,...,5.603286,0.585521,0.129345,1.529046,0.777626,0.535117,0.813652,CHIJ St. Nicholas Girls',1,0
3,2017-01-01,ANG MO KIO,3 ROOM,465,ANG MO KIO AVE 10,04 TO 06,68.0,New Generation,1980,62 years 01 month,...,5.599856,0.245970,0.069453,0.891974,0.697427,0.945527,1.952040,Catholic High,0,1
4,2017-01-01,ANG MO KIO,3 ROOM,601,ANG MO KIO AVE 5,01 TO 03,67.0,New Generation,1980,62 years 05 months,...,5.614671,0.611017,0.149898,1.572922,0.782771,0.501151,0.846780,CHIJ St. Nicholas Girls',1,0


In [16]:
# save to csv
merged_df.to_csv("../data/cleaned/merged.csv", index=False)